In [ ]:
## Data Source and Description

**Dataset:** Polish Companies Bankruptcy Data  
**Source:** UCI Machine Learning Repository  
**URL:** https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data

The dataset contains financial information for Polish companies obtained from the Emerging Markets Information Service (EMIS). It includes 64 financial ratios describing profitability, liquidity, leverage, and operating performance, along with a binary indicator identifying whether a company subsequently entered bankruptcy.

The original dataset contains 43,405 observations organized into five subsets representing different bankruptcy prediction horizons, ranging from one to five years.

### Project Dataset Selection

This project focuses on predicting bankruptcy **two years in advance**, corresponding to `year == 4` in the UCI dataset.

The selected subset contains:

- **9,792 observations**
- **64 financial predictors**
- **515 bankrupt companies (5.26%)**
- **9,277 non-bankrupt companies (94.74%)**

The `year` variable is excluded from the model predictors because it identifies the forecasting horizon rather than a financial characteristic of the company.

### Data Acquisition and Preparation

The dataset was retrieved using the Python `ucimlrepo` package (`fetch_ucirepo(id=365)`).

The initial import notebook performs the following operations:

1. Retrieves the complete UCI dataset.
2. Examines the dataset dimensions, class distribution, and missing values.
3. Selects observations corresponding to the two-year prediction horizon.
4. Removes the `year` identifier and resets the index.
5. Saves the selected observations to `data/processed/bankruptcy_2yr.csv`.

No missing-value imputation, scaling, outlier treatment, or feature selection is performed at this stage. These operations will be evaluated during exploratory analysis and model development, with fitted preprocessing restricted to training data to avoid data leakage.

### Reference

Zięba, M., Tomczak, S. K., and Tomczak, J. M. (2016). *Ensemble Boosted Trees with Synthetic Features Generation in Application to Bankruptcy Prediction*. Expert Systems with Applications, 58, 93–101.

UCI Machine Learning Repository: *Polish Companies Bankruptcy Data*, Dataset ID 365.

### Data Retrieval

In [ ]:
pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
polish_companies_bankruptcy = fetch_ucirepo(id=365) 
  
# data (as pandas dataframes) 
X = polish_companies_bankruptcy.data.features 
y = polish_companies_bankruptcy.data.targets 
  
# metadata 
print(polish_companies_bankruptcy.metadata) 
  
# variable information 
print(polish_companies_bankruptcy.variables) 


{'uci_id': 365, 'name': 'Polish Companies Bankruptcy', 'repository_url': 'https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data', 'data_url': 'https://archive.ics.uci.edu/static/public/365/data.csv', 'abstract': 'The dataset is about bankruptcy prediction of Polish companies.The bankrupt companies were analyzed in the period 2000-2012, while the still operating companies were evaluated from 2007 to 2013.', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 10503, 'num_features': 65, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2016, 'last_updated': 'Sat Mar 09 2024', 'dataset_doi': '10.24432/C5F600', 'creators': ['Sebastian Tomczak'], 'intro_paper': {'ID': 417, 'type': 'NATIVE', 'title': 'Ensemble boosted trees with synthetic features generation in application to bankruptcy predictio

### Initial Inspection - Dimensions, columns, class distribution

In [3]:

import pandas as pd

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts(dropna=False))

if "year" in X.columns:
    print("\nYear distribution:")
    print(X["year"].value_counts().sort_index())

print("\nMissing values:")
print(X.isna().sum().sort_values(ascending=False).head(15))


X shape: (43405, 65)
y shape: (43405, 1)

Feature columns:
['year', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63', 'A64']

Target distribution:
class
0        41314
1         2091
Name: count, dtype: int64

Year distribution:
year
1     7027
2    10173
3    10503
4     9792
5     5910
Name: count, dtype: int64

Missing values:
A37    18984
A21     5854
A27     2764
A60     2152
A45     2147
A24      922
A64      812
A54      812
A28      812
A53      812
A41      754
A32      368
A52      301
A47      297
A46      135
dtype: int64


In [12]:

# Combine features and target for descriptive analysis
df = X.copy()
df["class"] = y["class"]

# Summary by forecasting horizon
summary = (
    df.groupby("year")["class"]
      .agg(
          observations="count",
          bankruptcies="sum",
          bankruptcy_rate="mean"
      )
)

summary["non_bankrupt"] = (
    summary["observations"] - summary["bankruptcies"]
)

summary["bankruptcy_rate"] *= 100

print(summary.round(2))


# save raw dataset
from pathlib import Path

repo_root = Path.cwd().parent
relative_path = Path("data/raw/bankruptcy_all.csv")

output_path = repo_root / relative_path
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

      observations  bankruptcies  bankruptcy_rate  non_bankrupt
year                                                           
1             7027           271             3.86          6756
2            10173           400             3.93          9773
3            10503           495             4.71         10008
4             9792           515             5.26          9277
5             5910           410             6.94          5500


### Data Quality Overview - Missing value summary

In [5]:

missing_by_year = (
    df.groupby("year")
      .apply(
          lambda group: group.isna().mean() * 100,
          include_groups=False
      )
)

print(missing_by_year[["A37", "A21", "A27", "A60", "A45"]].round(2))


        A37    A21   A27   A60   A45
year                                
1     38.99  23.08  4.43  1.92  1.91
2     44.41  31.10  6.94  5.34  5.32
3     45.09   7.68  6.81  5.64  5.63
4     45.36   1.61  6.55  6.27  6.26
5     43.11   1.74  6.62  4.53  4.53


### Forecast Horizon Selection: Select year == 4

In [8]:

# Select the two-year bankruptcy forecasting horizon
mask = X["year"] == 4

# Exclude 'year' because it is constant in this subset
X_2yr = X.loc[mask].drop(columns=["year"]).copy()
y_2yr = y.loc[mask, "class"].copy()

# Reset indices for subsequent processing
X_2yr = X_2yr.reset_index(drop=True)
y_2yr = y_2yr.reset_index(drop=True)

print("Features:", X_2yr.shape)
print("Target:", y_2yr.shape)
print("\nClass distribution:")
print(y_2yr.value_counts())
print("\nBankruptcy rate:", f"{y_2yr.mean():.2%}")


Features: (9792, 64)
Target: (9792,)

Class distribution:
class
0    9277
1     515
Name: count, dtype: int64

Bankruptcy rate: 5.26%


### Export: Save selected CSV using relative path

In [10]:

repo_root = Path.cwd().parent
relative_path = Path("data/processed/bankruptcy_2yr.csv")

output_path = repo_root / relative_path
output_path.parent.mkdir(parents=True, exist_ok=True)

df_2yr.to_csv(output_path, index=False)

print(f"Saved: {relative_path.as_posix()}")
print(f"Shape: {df_2yr.shape}")



Saved: data/processed/bankruptcy_2yr.csv
Shape: (9792, 65)
